# Beijing PM2.5 — Notebook

Generated by `scripts/build_notebook.py` and executed end to end.

1. Load the 43,824 hourly records and validate continuity
2. Inspect the missing-target runs and why they are not imputed
3. Profile the seasonal, diurnal and wind-driven structure
4. Review the 2014 hold-out scores against persistence


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from airquality import config, data
from airquality.clean import clean
from airquality.features import add_features

pd.set_option("display.max_columns", 50)


## 1. Load and validate

In [2]:
raw = data.load_raw()
validation = data.validate(raw)
print(f"rows={validation['rows']:,} missing={validation['missing_target']:,} "
      f"({validation['missing_share']:.2%})")
print(f"episodes >150: {validation['episode_hours']:,} "
      f"({validation['episode_share']:.1%})")
pd.Series(validation["checks"]).apply(lambda row: row["detail"], convert_dtype=False)


rows=43,824 missing=2,067 (4.72%)
episodes >150: 8,991 (20.5%)


C:\Users\manis\AppData\Local\Temp\ipykernel_30140\4013086891.py:7: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  pd.Series(validation["checks"]).apply(lambda row: row["detail"], convert_dtype=False)


0                           all 13 columns present
1                                      43,824 rows
2     43,824 rows (expected 43,824 hourly records)
3                           0 duplicate timestamps
4                      records are ordered in time
5        0 gaps larger than one hour in the series
6                              pm2.5 within 0-1100
7                  missing values: {'pm2.5': 2067}
8     4.72% of target values missing (2,067 hours)
9                                 NW/NE/SE/cv only
10               first record: 2010-01-01 00:00:00
11                last record: 2014-12-31 23:00:00
dtype: object

## 2. Cleaning and missing runs

In [3]:
cleaned, log = clean(raw)
print("longest missing runs (hours):", log["missing_runs"]["longest_runs"])
print("imputation policy:", log["imputation_policy"])
pd.DataFrame(log["yearly"]).T


longest missing runs (hours): [155, 127, 99, 91, 76]
imputation policy: no imputation: a missing target cannot be trained on, and interpolating it would fake the very spikes the model is meant to catch


,mean,median,hours
2010,104.05,79.0,8091.0
2011,99.07,71.0,8032.0
2012,90.55,69.0,8295.0
2013,101.71,71.5,8678.0
2014,97.73,72.0,8661.0


## 3. Structure of the series

In [4]:
engineered = add_features(cleaned)
engineered.groupby("season")["pm2.5"].agg(["mean", "median", "size"]).round(2)


,mean,median,size
season,,,
Autumn,104.22,73.0,10920
Spring,87.21,69.0,11040
Summer,90.44,77.0,11040
Winter,112.78,70.0,10824


In [5]:
engineered.groupby("cbwd")["pm2.5"].agg(["mean", "size"]).round(2)


,mean,size
cbwd,,
NE,90.18,4997
NW,70.13,14150
SE,110.82,15290
cv,126.15,9387


In [6]:
engineered.groupby("hour")["pm2.5"].mean().round(1).to_frame("mean_pm25")


,mean_pm25
hour,
0,113.4
1,113.7
2,110.3
3,108.0
4,104.1
5,100.0
6,96.9
7,96.0
8,95.9


## 4. Hold-out results

Read from `reports/metrics.json`.

In [7]:
import json

path = ROOT / "reports" / "metrics.json"
metrics = json.loads(path.read_text()) if path.exists() else {}

if metrics:
    display(pd.DataFrame({
        name: block["test"] for name, block in metrics["scenarios"].items()
    }))
    display(pd.Series(metrics["baselines"]["persistence"]).to_frame("persistence"))
    display(pd.DataFrame(metrics["feature_importance"]).round(4))
else:
    print("Run `python run.py train` first.")


,weather_only,with_lags
rmse,69.0560,21.2430
mae,47.0610,12.0190
r2,0.4548,0.9484
bias,-2.5310,-0.4100
within_25,0.4316,0.8923
hours,8661.0000,8266.0000


,persistence
rmse,22.0910
mae,11.9220
r2,0.9442
bias,-0.0680
within_25,0.8914
hours,8266.0000


,feature,importance
0,pm_lag_1,62.9594
1,pm_diff_24,31.3904
2,pm_lag_24,28.0278
3,pm_lag_3,8.3631
4,pm_roll_24,4.2946
5,log_wind,3.9404
6,cbwd_SE,3.6079
7,hour_sin,3.5416
8,dew_point_depression,3.3562
9,cbwd_NW,3.3526


## 5. Reading the result

The `with_lags` scenario is only a small improvement over the persistence baseline,
and the weather-only model has genuine but modest skill. The honest headline is that
meteorology explains the envelope, not the spikes.
